

# Parametric vs. Nonparametric Monte Carlo
### OPIM 5641 - Business Decision Modeling · Module 1.2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/2_MonteCarlo/Parametric_vs_Nonparametric_MC.ipynb)

*Run me top to bottom - **Runtime → Run all**. The data loads from the class repo, so there's nothing to upload.*

One retirement question, answered two ways. Same starting balance, same contributions, same 30 years, same 10,000 trials - the ONLY thing that changes is where the randomness comes from:

- **Parametric:** draw each year's return from a **named distribution** (a Normal whose mean and standard deviation we estimate from real S&P history)
- **Nonparametric:** skip the distribution entirely and **resample actual years from that same history**, with replacement

🔷 **The nugget:** *you* are the modeler - parametric vs. nonparametric isn't right-or-wrong, it's a judgment call about what you actually know. This notebook exists so you can SEE the consequences of that call, holding everything else fixed.

## Our assumptions
Same table for BOTH methods - that's the whole point. Change one thing (the sampling method), hold everything else constant, and any difference in the answers is attributable to that one change.

| Assumption | Our choice |
|---|---|
| Starting balance | **\$100,000** |
| Contribution | **\$10,000 per year, flat** (same for both methods!) |
| Horizon | **30 years** |
| Trials | **10,000 careers each** |
| Where returns come from | parametric: Normal(mean, sd) estimated from ~95 years of S&P history · nonparametric: resample those same years, with replacement |
| Taxes, fees, inflation | ignored (a modeling choice, not a law of nature) |
| Seed | `5641` - so your run reproduces mine; comment it out to feel the randomness |

In [ ]:
# import our libraries - every line has a comment, that's the house style!
import pandas as pd               # tables
import numpy as np                # random draws + math
import matplotlib.pyplot as plt   # plots
from matplotlib.ticker import FuncFormatter # dollar-format the axes

np.random.seed(5641) # reproducible randomness - comment out and re-run to watch the numbers wiggle

# The fuel: ~95 years of real S&P returns
Both methods run on the SAME history - the parametric method just compresses it into two numbers first.

In [ ]:
# annual S&P 500 percent changes, hosted right in the class repo (no Drive, no gdown)
df = pd.read_csv('https://raw.githubusercontent.com/drdave-teaching/OPIM5641-notebooks/main/2_MonteCarlo/data/S_P_returns.csv')

# the percent column has a % character - strip it and convert (sound familiar?)
df['percChange'] = pd.to_numeric(df['percChange'].str.replace('%', '', regex=False), errors='coerce')
df['percChange'].hist(bins=20, color='steelblue', edgecolor='black')
plt.title('Annual S&P 500 Returns - the real history both methods feed on')
plt.xlabel('Annual return (%)')
plt.ylabel('Number of years')
plt.show()

In [ ]:
# the parametric method compresses ALL that history into just two numbers:
rate_mean = df['percChange'].mean() # the average annual return, %
rate_sd = df['percChange'].std()    # the volatility, %
print(f'mean = {rate_mean:.2f}%   sd = {rate_sd:.2f}%')
# everything else about the shape - the crash years, the streaks - gets thrown away. Remember that.

# Method 1 - Parametric
Every year's growth rate is a fresh draw from `Normal(rate_mean, rate_sd)`. We save the **whole 30-year path** of every trial so we can draw spaghetti.

In [ ]:
# 10,000 parametric careers
paths = [] # one entry per trial - the full 30-year trajectory
for b in np.arange(0,10000,1):
  initialMoney = 100000 # starting balance
  path = [] # this trial's year-by-year balances
  rates = 1 + np.random.normal(loc=rate_mean, scale=rate_sd, size=30)/100 # 30 years of returns in one call
  for a in np.arange(0,30,1):
    contrib = 10000 # annual contribution, $ - flat, same as nonparametric
    savings = (initialMoney + contrib)*rates[a] # deposit, then grow
    initialMoney = savings # the handoff: this year's ending balance starts next year
    path.append(float(savings))
  paths.append(pd.Series(path, name=b))
pathsDF_param = pd.concat(paths, axis=1) # 30 rows (years) x 10,000 columns (trials)
final_param = pathsDF_param.iloc[-1] # everyone's year-30 balance
final_param.describe()

In [ ]:
# spaghetti - all 10,000 parametric futures
plt.figure(figsize=(10,6))
plt.plot(pathsDF_param, color='steelblue', alpha=0.02, linewidth=0.5) # 10,000 strands
plt.plot(pathsDF_param.quantile(0.95, axis=1), color='green', linewidth=2, linestyle='--', label='95th percentile')
plt.plot(pathsDF_param.mean(axis=1), color='purple', linewidth=2, label='mean path')
plt.plot(pathsDF_param.median(axis=1), color='black', linewidth=2, label='median path')
plt.plot(pathsDF_param.quantile(0.05, axis=1), color='red', linewidth=2, linestyle='--', label='5th percentile')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M'))
plt.title('Parametric - 10,000 Simulated Futures')
plt.xlabel('Year')
plt.ylabel('Savings')
plt.legend()
plt.show()

# Method 2 - Nonparametric
No distribution, no assumptions about shape. Each simulated year, we reach into the bag of ~95 REAL years and pull one out (**with replacement** - the same year can be drawn twice). History speaks for itself, crash years and all.

In [ ]:
# 10,000 nonparametric careers - identical code except ONE line
paths = []
for b in np.arange(0,10000,1):
  initialMoney = 100000 # starting balance - same
  path = []
  rates = 1 + df['percChange'].sample(n=30, replace=True).values/100 # <-- THE line: resample real history
  for a in np.arange(0,30,1):
    contrib = 10000 # flat contribution - same
    savings = (initialMoney + contrib)*rates[a]
    initialMoney = savings
    path.append(float(savings))
  paths.append(pd.Series(path, name=b))
pathsDF_np = pd.concat(paths, axis=1)
final_np = pathsDF_np.iloc[-1]
final_np.describe()

In [ ]:
# spaghetti - all 10,000 nonparametric futures
plt.figure(figsize=(10,6))
plt.plot(pathsDF_np, color='darkorange', alpha=0.02, linewidth=0.5)
plt.plot(pathsDF_np.quantile(0.95, axis=1), color='green', linewidth=2, linestyle='--', label='95th percentile')
plt.plot(pathsDF_np.mean(axis=1), color='purple', linewidth=2, label='mean path')
plt.plot(pathsDF_np.median(axis=1), color='black', linewidth=2, label='median path')
plt.plot(pathsDF_np.quantile(0.05, axis=1), color='red', linewidth=2, linestyle='--', label='5th percentile')
plt.gca().yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1e6:.1f}M'))
plt.title('Nonparametric - 10,000 Simulated Futures (resampled real history)')
plt.xlabel('Year')
plt.ylabel('Savings')
plt.legend()
plt.show()

# Head to head
Same assumptions, same seed, same number of trials - one line of code different. Let the numbers and the overlaid histograms tell the story.

In [ ]:
# side-by-side summary - the client-facing numbers
compare = pd.DataFrame({
    'Parametric': [final_param.mean(), final_param.median(), final_param.quantile(0.05), final_param.quantile(0.95)],
    'Nonparametric': [final_np.mean(), final_np.median(), final_np.quantile(0.05), final_np.quantile(0.95)],
}, index=['Mean', 'Median', '5th percentile', '95th percentile'])
(compare/1e6).round(2) # in $ millions

In [ ]:
# overlay the two outcome distributions
plt.figure(figsize=(10,6))
plt.hist(final_param, bins=100, alpha=0.5, color='steelblue', label='Parametric')
plt.hist(final_np, bins=100, alpha=0.5, color='darkorange', label='Nonparametric')
plt.gca().xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'${x/1e6:.0f}M'))
plt.title('Retirement Outcomes - Parametric vs. Nonparametric')
plt.xlabel('Savings at year 30')
plt.ylabel('Number of trials')
plt.legend()
plt.show()

# So which one should you use?
The two methods land close - and they SHOULD, because the Normal was estimated *from* the very history the nonparametric method resamples. The differences live in the details, and the details are the judgment call:

**Choose parametric when...**
- you want a *smooth* model you can reason about ("what if volatility rises to 25%?" - just turn the knob)
- you need values history never produced - the Normal can draw a year worse than 1931, real history can't
- you have expert opinion but thin data (that's where the triangular distribution shines, too)

**Choose nonparametric when...**
- you distrust the shape assumption - real returns have **fatter tails and skew** than a Normal admits
- the data IS the argument: "these are actual years that actually happened" lands with clients
- you have plenty of history and no reason to smooth it

**Caution:** each method hides a different risk. The Normal quietly *thins the tails* - it makes catastrophic years look rarer than history says. Resampling quietly *caps the tails* - it can never produce anything worse than the worst year on record. Neither one is "the safe choice." Say out loud which risk you're accepting.

**On your own:**
1. Make the contribution escalate 3% per year in BOTH methods. How much does the median retirement improve?
2. Comment out the seed and run the notebook three times. How much do the compare-table numbers move? Does that change any *decision*?
3. Parametric with fatter tails: swap the Normal for a Student-t (`np.random.standard_t(df=5, size=30)` scaled by `rate_sd`, shifted by `rate_mean`). Which histogram does it resemble now?

------------------------------------------------

**Bottom line:** one line of code separates the two philosophies - `np.random.normal(...)` versus `.sample(replace=True)`. Everything else - the loop, the handoff, the percentile thinking - is identical. Master the recipe once and the sampling method becomes what it should be: a *choice you defend*, not a default you inherit.